# MMDLQA Colab Quickstart

Run cells from top to bottom. Recommended workflow: keep code in GitHub, keep only `input/questions.xlsx` and output files in Google Drive. Colab clones/pulls code into `/content/MMDLQA`, downloads `raw_1.zip`, `raw_2.zip`, and `text_cleaning_output.zip` with `gdown`, then uses local runtime storage for the heavy data.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone Or Pull Project Code

Edit `REPO_URL` to your GitHub repository. Edit `DRIVE_DATA_DIR` to the Drive folder that contains only `input/questions.xlsx` and where output should be written. Paste the Google Drive share links or file ids for the three dataset zip files below.

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/HuyThinhUET/MMDLQA.git'
PROJECT_DIR = '/content/MMDLQA'
DRIVE_DATA_DIR = '/content/drive/MyDrive/MMDLQA_data'
LOCAL_DATA_DIR = '/content/MMDLQA_data_runtime'
QUESTION_FILENAME = 'questions.xlsx'  # change to 'question.xlsx' if your Drive file uses that name

# Paste Google Drive share links or file ids here. Leave blank only if the file already exists locally.
RAW_1_GDOWN = ''
RAW_2_GDOWN = ''
TEXT_CLEANING_OUTPUT_GDOWN = ''

if not Path(PROJECT_DIR, '.git').exists():
    !git clone $REPO_URL $PROJECT_DIR
else:
    !git -C $PROJECT_DIR pull

os.chdir(PROJECT_DIR)
os.environ['MMDLQA_RAW_DIR'] = f'{LOCAL_DATA_DIR}/input/raw'
os.environ['MMDLQA_TEXT_CLEANING_OUTPUT_DIR'] = f'{LOCAL_DATA_DIR}/input/text_cleaning_output'
os.environ['MMDLQA_USE_TEXT_CLEANING_OUTPUT'] = '1'
os.environ['MMDLQA_INCLUDE_RAW_FALLBACK'] = '1'
question_candidates = [Path(DRIVE_DATA_DIR) / 'input' / QUESTION_FILENAME]
if QUESTION_FILENAME != 'question.xlsx':
    question_candidates.append(Path(DRIVE_DATA_DIR) / 'input' / 'question.xlsx')
os.environ['MMDLQA_QUESTIONS'] = str(next((p for p in question_candidates if p.exists()), question_candidates[0]))
os.environ['MMDLQA_OUTPUT_DIR'] = f'{DRIVE_DATA_DIR}/output'
os.environ['MMDLQA_CACHE_DIR'] = f'{LOCAL_DATA_DIR}/cache'
os.environ['MMDLQA_SUBMISSION'] = f'{DRIVE_DATA_DIR}/output/submission.csv'
for key in ['MMDLQA_RAW_DIR', 'MMDLQA_TEXT_CLEANING_OUTPUT_DIR', 'MMDLQA_OUTPUT_DIR', 'MMDLQA_CACHE_DIR']:
    Path(os.environ[key]).mkdir(parents=True, exist_ok=True)

!pwd
!git status --short
print('local data:', LOCAL_DATA_DIR)
print('raw:', os.environ['MMDLQA_RAW_DIR'])
print('cleaned:', os.environ['MMDLQA_TEXT_CLEANING_OUTPUT_DIR'])
print('questions:', os.environ['MMDLQA_QUESTIONS'])
print('output:', os.environ['MMDLQA_OUTPUT_DIR'])

## 3. Install Dependencies

This can take a few minutes, especially Whisper.

In [ ]:
!apt-get update -y
!apt-get install -y ffmpeg tesseract-ocr tesseract-ocr-vie tesseract-ocr-chi-sim libreoffice poppler-utils
!pip install -r requirements.txt gdown

## 4. Download And Check Data Layout

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

import gdown

raw_dir = Path(os.environ['MMDLQA_RAW_DIR'])
cleaned_dir = Path(os.environ['MMDLQA_TEXT_CLEANING_OUTPUT_DIR'])
downloads_dir = Path(LOCAL_DATA_DIR) / 'downloads'
downloads_dir.mkdir(parents=True, exist_ok=True)

def gdown_download(source, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if output_path.exists() and output_path.stat().st_size > 0:
        print('exists:', output_path)
        return
    if not source:
        print('missing gdown link/id for', output_path.name)
        return
    source = str(source).strip()
    print('downloading:', output_path.name)
    if source.startswith('http://') or source.startswith('https://'):
        gdown.download(url=source, output=str(output_path), fuzzy=True, quiet=False)
    else:
        gdown.download(id=source, output=str(output_path), quiet=False)
    assert output_path.exists() and output_path.stat().st_size > 0, f'Download failed: {output_path}'

def materialize_text_cleaning_output(zip_path, cleaned_dir):
    zip_path = Path(zip_path)
    if not zip_path.exists():
        return
    if (cleaned_dir / 'by_file').exists():
        print('cleaned output exists:', cleaned_dir)
        return
    tmp = Path(LOCAL_DATA_DIR) / 'tmp_text_cleaning_output'
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(tmp)
    candidates = [tmp / 'text_cleaning_output', tmp / 'input' / 'text_cleaning_output', tmp]
    source_dir = next((p for p in candidates if (p / 'by_file').exists()), None)
    assert source_dir is not None, 'text_cleaning_output.zip must contain by_file/'
    if cleaned_dir.exists():
        shutil.rmtree(cleaned_dir)
    shutil.copytree(source_dir, cleaned_dir)
    print('materialized cleaned output:', cleaned_dir)

gdown_download(RAW_1_GDOWN, raw_dir / 'raw_1.zip')
gdown_download(RAW_2_GDOWN, raw_dir / 'raw_2.zip')
clean_zip = downloads_dir / 'text_cleaning_output.zip'
gdown_download(TEXT_CLEANING_OUTPUT_GDOWN, clean_zip)
materialize_text_cleaning_output(clean_zip, cleaned_dir)

!find "$MMDLQA_RAW_DIR" -type f | head -30
!python - <<'PY'
import os
from pathlib import Path
from collections import Counter
raw = Path(os.environ['MMDLQA_RAW_DIR'])
cleaned = Path(os.environ['MMDLQA_TEXT_CLEANING_OUTPUT_DIR'])
questions = Path(os.environ['MMDLQA_QUESTIONS'])
files = list(raw.rglob('*'))
files = [p for p in files if p.is_file()]
manifest = cleaned / 'text_cleaning_manifest.csv'
print('files:', len(files))
print(Counter(p.suffix.lower() for p in files).most_common())
print('cleaned exists:', cleaned.exists(), cleaned)
print('cleaned by_file dirs:', len(list((cleaned / 'by_file').glob('*'))) if (cleaned / 'by_file').exists() else 0)
print('cleaned manifest exists:', manifest.exists(), manifest)
print('questions exists:', questions.exists(), questions)

## 5. Set OpenRouter Key

Use Colab Secrets if possible: left sidebar -> key icon -> add `OPENROUTER_API_KEY`. If not, paste it below for a quick test.

In [ ]:
import os

def clean_openrouter_key(value):
    key = (value or '').strip().strip('"').strip("'")
    if key.lower().startswith('bearer '):
        key = key[7:].strip()
    key = key.replace('\\r', '').replace('\\n', '').replace('\\t', '')
    return ''.join(key.split())

try:
    from google.colab import userdata
    os.environ['OPENROUTER_API_KEY'] = clean_openrouter_key(userdata.get('OPENROUTER_API_KEY'))
except Exception:
    pass

# Uncomment this line only for a quick local test if you do not use Colab Secrets.
# os.environ['OPENROUTER_API_KEY'] = 'PASTE_YOUR_KEY_HERE'
os.environ['OPENROUTER_API_KEY'] = clean_openrouter_key(os.environ.get('OPENROUTER_API_KEY'))

os.environ['MMDLQA_USE_MODEL_ROUTER'] = '1'
os.environ['MMDLQA_PRINT_QUESTION_METRICS'] = '1'
os.environ['MMDLQA_USE_QUESTION_CLASSIFIER'] = '1'
os.environ['MMDLQA_USE_EVIDENCE_SCANNER'] = '1'
os.environ['MMDLQA_FORCE_BEST_EFFORT_ANSWER'] = '1'
os.environ['MMDLQA_EVIDENCE_SCAN_MAX_FILES'] = '12'
os.environ['MMDLQA_EVIDENCE_SCAN_IRRELEVANT_PATIENCE'] = '5'
os.environ['MMDLQA_RERANK_CANDIDATE_K'] = '24'
os.environ['MMDLQA_RERANK_TOP_K'] = '8'
os.environ['MMDLQA_AGENTIC_MIN_ROUNDS'] = '2'
os.environ['MMDLQA_AGENTIC_MAX_ROUNDS'] = '10'
os.environ['MMDLQA_MAX_QUESTIONS'] = '5'
os.environ['MMDLQA_MAX_QUESTION_COST_USD'] = '0.08'
os.environ['MMDLQA_PLANNER_MODEL'] = 'google/gemini-2.5-flash-lite'
os.environ['MMDLQA_RERANK_MODEL'] = 'google/gemini-2.5-flash-lite'
os.environ['MMDLQA_EXACT_MODEL'] = 'deepseek/deepseek-chat-v3.1'
os.environ['MMDLQA_SYNTHESIS_MODEL'] = 'deepseek/deepseek-chat-v3.1'
os.environ['MMDLQA_CRITIC_MODEL'] = 'deepseek/deepseek-chat-v3.1'
os.environ['MMDLQA_CODER_MODEL'] = 'qwen/qwen3-coder-flash'
os.environ['MMDLQA_VISION_MODEL'] = 'google/gemini-2.5-flash'
os.environ['MMDLQA_SCAN_TEXT_MODEL'] = 'google/gemini-2.5-flash-lite'
os.environ['MMDLQA_SCAN_TABLE_MODEL'] = 'qwen/qwen3-coder-flash'
os.environ['MMDLQA_SCAN_DOCUMENT_MODEL'] = 'deepseek/deepseek-chat-v3.1'
os.environ['MMDLQA_SCAN_IMAGE_MODEL'] = 'google/gemini-2.5-flash'
os.environ['MMDLQA_SCAN_AUDIO_MODEL'] = 'google/gemini-2.5-flash-lite'
os.environ['MMDLQA_SCAN_VIDEO_MODEL'] = 'google/gemini-2.5-flash-lite'
print('key set:', bool(os.environ.get('OPENROUTER_API_KEY')), 'length:', len(os.environ.get('OPENROUTER_API_KEY', '')))
print('planner model:', os.environ['MMDLQA_PLANNER_MODEL'])
print('synthesis model:', os.environ['MMDLQA_SYNTHESIS_MODEL'])

## 6. Cheap Baseline Dry Run

No OpenRouter calls. This verifies ingestion, indexing, retrieval, and CSV output with the baseline runner.

In [ ]:
%env MMDLQA_USE_LLM=0
%env MMDLQA_USE_WHISPER=0
%env MMDLQA_USE_VISION_LLM=0
!python script/run_pipeline.py --rebuild-index --text-cleaning-output "$MMDLQA_TEXT_CLEANING_OUTPUT_DIR"
!python script/evaluate_sample.py

## 7. Agentic Dry Run

No OpenRouter calls. This verifies the multi-agent workflow, sentence-level RAG boundary, deterministic tools, static critic, and diagnostics.

In [ ]:
%env MMDLQA_USE_LLM=0
%env MMDLQA_USE_WHISPER=0
%env MMDLQA_USE_VISION_LLM=0
%env MMDLQA_USE_AGENTIC_MOE=0
%env MMDLQA_USE_AGENTIC_CRITIC=0
!python script/run_agentic.py --rebuild-index --max-questions 5 --text-cleaning-output "$MMDLQA_TEXT_CLEANING_OUTPUT_DIR"
!python script/evaluate_sample.py

## 8. Agentic Run With LLM

This uses OpenRouter for planner/MoE/critic calls. Default `MMDLQA_MAX_QUESTIONS=5`; set it to `-1` to run the full questions file. Keep `MMDLQA_USE_LLM_SUMMARIES=0` first to avoid captioning every image during indexing.

In [ ]:
%env MMDLQA_USE_LLM=1
%env MMDLQA_USE_LLM_RERANK=1
%env MMDLQA_USE_VISION_LLM=1
%env MMDLQA_USE_LLM_SUMMARIES=0
%env MMDLQA_USE_WHISPER=1
%env MMDLQA_USE_AGENTIC_PLANNER=1
%env MMDLQA_USE_AGENTIC_MOE=1
%env MMDLQA_USE_AGENTIC_CRITIC=1
%env MMDLQA_USE_AGENTIC_TOOLS=1
%env MMDLQA_USE_AGENTIC_CODER=1
%env MMDLQA_USE_QUESTION_CLASSIFIER=1
%env MMDLQA_USE_EVIDENCE_SCANNER=1
%env MMDLQA_FORCE_BEST_EFFORT_ANSWER=1
%env MMDLQA_USE_CODER_PLANNER=0
%env MMDLQA_USE_MODEL_ROUTER=1
%env MMDLQA_PRINT_QUESTION_METRICS=1
%env MMDLQA_PLANNER_MODEL=google/gemini-2.5-flash-lite
%env MMDLQA_RERANK_MODEL=google/gemini-2.5-flash-lite
%env MMDLQA_EXACT_MODEL=deepseek/deepseek-chat-v3.1
%env MMDLQA_SYNTHESIS_MODEL=deepseek/deepseek-chat-v3.1
%env MMDLQA_CRITIC_MODEL=deepseek/deepseek-chat-v3.1
%env MMDLQA_CODER_MODEL=qwen/qwen3-coder-flash
%env MMDLQA_VISION_MODEL=google/gemini-2.5-flash
%env MMDLQA_SCAN_TEXT_MODEL=google/gemini-2.5-flash-lite
%env MMDLQA_SCAN_TABLE_MODEL=qwen/qwen3-coder-flash
%env MMDLQA_SCAN_DOCUMENT_MODEL=deepseek/deepseek-chat-v3.1
%env MMDLQA_SCAN_IMAGE_MODEL=google/gemini-2.5-flash
%env MMDLQA_SCAN_AUDIO_MODEL=google/gemini-2.5-flash-lite
%env MMDLQA_SCAN_VIDEO_MODEL=google/gemini-2.5-flash-lite
%env MMDLQA_AGENTIC_MAX_STEPS=4
%env MMDLQA_AGENTIC_MIN_ROUNDS=2
%env MMDLQA_AGENTIC_MAX_ROUNDS=10
%env MMDLQA_EVIDENCE_SCAN_MAX_FILES=12
%env MMDLQA_EVIDENCE_SCAN_CHUNKS_PER_FILE=2
%env MMDLQA_EVIDENCE_SCAN_BATCH_SIZE=12
%env MMDLQA_EVIDENCE_SCAN_IRRELEVANT_PATIENCE=5
%env MMDLQA_RERANK_CANDIDATE_K=24
%env MMDLQA_RERANK_TOP_K=8
%env MMDLQA_MAX_CONTEXT_CHARS=14000
%env MMDLQA_MAX_QUESTIONS=5
%env MMDLQA_MAX_QUESTION_SECONDS=0
%env MMDLQA_MAX_QUESTION_LLM_CALLS=0
%env MMDLQA_MAX_QUESTION_COST_USD=0.08
%env MMDLQA_MAX_QUESTION_RAG_QUERIES=0
%env MMDLQA_LLM_INPUT_COST_PER_MILLION_TOKENS=0
%env MMDLQA_LLM_OUTPUT_COST_PER_MILLION_TOKENS=0
!python -c "import os; from mmdlqa_core.openrouter import sanitize_api_key; k=sanitize_api_key(os.environ.get('OPENROUTER_API_KEY', '')); assert os.environ.get('MMDLQA_USE_LLM') == '1', 'MMDLQA_USE_LLM must be 1 for LLM run'; assert k, 'OPENROUTER_API_KEY is missing'; print('LLM preflight ok, key length:', len(k))"
!python script/run_agentic.py --rebuild-index --max-questions "$MMDLQA_MAX_QUESTIONS" --text-cleaning-output "$MMDLQA_TEXT_CLEANING_OUTPUT_DIR"
!python script/evaluate_sample.py

## 9. Inspect Cost And Time

The run cell prints live per-question progress when `MMDLQA_PRINT_QUESTION_METRICS=1`. This cell renders the final metrics table from `diagnostics.jsonl`.

In [ ]:
import json
import os
from pathlib import Path
import pandas as pd

out = Path(os.environ['MMDLQA_OUTPUT_DIR'])
summary_path = out / 'run_summary.json'
diagnostics_path = out / 'diagnostics.jsonl'

summary = json.loads(summary_path.read_text(encoding='utf-8'))
print('mode:', summary.get('mode'))
print('questions:', summary.get('questions'))
print('indexed:', summary.get('indexed_files'), 'files /', summary.get('indexed_chunks'), 'chunks')
print('total elapsed sec:', summary.get('metrics', {}).get('total_elapsed_sec'))
print('total llm calls:', summary.get('metrics', {}).get('total_llm_calls'))
print('total failed llm calls:', summary.get('metrics', {}).get('total_failed_llm_calls'))
print('total estimated cost usd:', summary.get('metrics', {}).get('total_estimated_cost_usd'))

rows = []
for line in diagnostics_path.read_text(encoding='utf-8').splitlines():
    if not line.strip():
        continue
    row = json.loads(line)
    answer = row.get('answer', {})
    metrics = answer.get('diagnostics', {}).get('metrics', {})
    agentic = answer.get('diagnostics', {}).get('agentic', {})
    scans = agentic.get('diagnostics', {}).get('evidence_scans', [])
    rows.append({
        'qid': answer.get('qid'),
        'elapsed_sec': metrics.get('elapsed_sec'),
        'llm_calls': metrics.get('llm_call_count'),
        'failed_llm': metrics.get('failed_llm_call_count'),
        'tokens': metrics.get('total_tokens'),
        'cost_usd': metrics.get('total_estimated_cost_usd'),
        'first_llm_error': str(metrics.get('first_llm_error', ''))[:160],
        'scan_batches': len(scans),
        'scanned_files': sum(s.get('processed_files', 0) for s in scans if isinstance(s, dict)),
        'answer': str(answer.get('answer', ''))[:120],
    })

df = pd.DataFrame(rows)
display(df)
display(df[['elapsed_sec', 'llm_calls', 'failed_llm', 'tokens', 'cost_usd', 'scanned_files']].describe())

## 10. Download Submission

In [ ]:
from google.colab import files
import os
files.download(os.environ['MMDLQA_SUBMISSION'])